# Scale Worm Detection — Model Metrics & Temporal Abundance
**Species:** *Lepidonotopodium piscesae* | **Site:** Mushroom Vent, Axial Seamount | **System:** CAMHD OOI

This notebook:
1. Unpacks your verified YOLO-format dataset zip
2. Parses image filenames to extract timestamps
3. Computes model performance metrics (precision, recall, F1, confusion matrix)
4. Generates interactive Plotly charts: worms over time + model metrics

## 1. Setup & Imports

In [3]:
import os
import re
import zipfile
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)
from datetime import datetime
from pathlib import Path

print('Libraries loaded successfully.')

ModuleNotFoundError: No module named 'plotly'

## 2. Configuration — Set Your Paths Here

In [1]:
# -------------------------------------------------------
# CONFIGURATION — edit these values if paths change
# -------------------------------------------------------
ZIP_PATH = '/home/jovyan/scaleworm_student_lab/notebooks/verification_session/verified_scaleworm_dataset_2024-10-01_to_2024-10-31.zip'
EXTRACT_DIR = '/home/jovyan/scaleworm_student_lab/notebooks/verification_session/extracted_dataset'

# Folder names inside the zip (adjust if yours differ)
POSITIVE_FOLDER = 'scaleworm'       # confirmed scale worm crops
NEGATIVE_FOLDER = 'not_scaleworm'   # rejected / false positive crops

# Regex to extract timestamp from filename
# Handles filenames like: frame_2024-10-03_12-05-30.jpg
#                         CAMHD_2024-10-15_08-22-11_crop.jpg
TIMESTAMP_REGEX = r'(\d{4}-\d{2}-\d{2})[_-](\d{2}-\d{2}-\d{2})'
TIMESTAMP_FORMAT = '%Y-%m-%d %H-%M-%S'

print(f'ZIP path   : {ZIP_PATH}')
print(f'Extract to : {EXTRACT_DIR}')

ZIP path   : /home/jovyan/scaleworm_student_lab/notebooks/verification_session/verified_scaleworm_dataset_2024-10-01_to_2024-10-31.zip
Extract to : /home/jovyan/scaleworm_student_lab/notebooks/verification_session/extracted_dataset


## 3. Unzip the Dataset

In [2]:
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

print('Extraction complete. Top-level contents:')
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    if level > 2:
        continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')
        if len(files) > 5:
            print(f'{indent}  ... and {len(files) - 5} more files')

NameError: name 'os' is not defined

## 4. Parse Filenames & Build Detection DataFrame

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}

def parse_timestamp(filename):
    """Extract datetime from filename. Returns None if no match."""
    match = re.search(TIMESTAMP_REGEX, filename)
    if match:
        ts_str = f'{match.group(1)} {match.group(2)}'
        try:
            return datetime.strptime(ts_str, TIMESTAMP_FORMAT)
        except ValueError:
            return None
    return None

records = []

for label, folder_name in [('scaleworm', POSITIVE_FOLDER), ('not_scaleworm', NEGATIVE_FOLDER)]:
    folder_path = Path(EXTRACT_DIR) / folder_name

    # Search one level deeper in case zip has a root wrapper folder
    if not folder_path.exists():
        matches = list(Path(EXTRACT_DIR).rglob(folder_name))
        if matches:
            folder_path = matches[0]

    if not folder_path.exists():
        print(f'WARNING: Could not find folder "{folder_name}" — check POSITIVE_FOLDER / NEGATIVE_FOLDER in config.')
        continue

    for img_file in folder_path.iterdir():
        if img_file.suffix.lower() in IMAGE_EXTENSIONS:
            ts = parse_timestamp(img_file.name)
            records.append({
                'filename': img_file.name,
                'verified_label': label,
                'is_scaleworm': 1 if label == 'scaleworm' else 0,
                'timestamp': ts,
                'folder': folder_name
            })

df = pd.DataFrame(records)
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Total images parsed       : {len(df)}')
print(f'  Confirmed scale worms   : {df["is_scaleworm"].sum()}')
print(f'  Not scale worms         : {(df["is_scaleworm"] == 0).sum()}')
print(f'  With parsed timestamps  : {df["timestamp"].notna().sum()}')

# Show a sample so you can verify parsing looks correct
df.head(10)

## 5. Model Metrics

> **How this works:** The YOLO Mushroom Model predicted a detection for every image in this dataset (that's why they exist as crops). Your verification sorted them into `scaleworm/` (model was right — true positive) and `not_scaleworm/` (model was wrong — false positive). The model's predicted label is therefore always **positive (1)**; your folder assignment is the ground truth.
>
> **Precision** = TP / (TP + FP) tells you what fraction of the model's detections were actually scale worms. As you correct and retrain, this number should rise.

In [ ]:
y_pred = np.ones(len(df), dtype=int)   # model flagged everything as a detection
y_true = df['is_scaleworm'].values

precision = precision_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)
tp  = int(np.sum((y_pred == 1) & (y_true == 1)))
fp  = int(np.sum((y_pred == 1) & (y_true == 0)))
total = len(df)

print('=== MODEL PERFORMANCE SUMMARY ===')
print(f'Total detections reviewed  : {total}')
print(f'True Positives  (TP)       : {tp}  — correctly identified scale worms')
print(f'False Positives (FP)       : {fp}  — flagged by model, not a scale worm')
print(f'Precision                  : {precision:.3f}  ({precision*100:.1f}% of detections were real worms)')
print(f'F1 Score                   : {f1:.3f}')
print()
print(classification_report(y_true, y_pred, target_names=['not_scaleworm', 'scaleworm']))

## 6. Interactive Model Metrics Charts

In [ ]:
# --- Confusion Matrix heatmap ---
cm = confusion_matrix(y_true, y_pred)

fig_cm = go.Figure(data=go.Heatmap(
    z=cm,
    x=['Predicted: Not Worm', 'Predicted: Worm'],
    y=['Actual: Not Worm', 'Actual: Worm'],
    colorscale='Teal',
    text=cm,
    texttemplate='%{text}',
    textfont={'size': 22},
    showscale=True
))
fig_cm.update_layout(
    title='Confusion Matrix — YOLO Mushroom Model',
    xaxis_title='Model Prediction',
    yaxis_title='Verified Ground Truth',
    template='plotly_dark',
    width=550, height=450
)
fig_cm.show()

In [ ]:
# --- TP vs FP bar chart ---
fig_bar = go.Figure(data=[
    go.Bar(name='True Positives',  x=['Detections'], y=[tp],
           marker_color='#2ec4b6', text=[tp], textposition='outside'),
    go.Bar(name='False Positives', x=['Detections'], y=[fp],
           marker_color='#e76f51', text=[fp], textposition='outside')
])
fig_bar.update_layout(
    title='True vs False Positive Detections',
    yaxis_title='Count',
    barmode='group',
    template='plotly_dark',
    width=500, height=420
)
fig_bar.show()

In [ ]:
# --- Precision gauge ---
fig_gauge = go.Figure(go.Indicator(
    mode='gauge+number',
    value=round(precision * 100, 1),
    title={'text': 'Model Precision (%)'},
    gauge={
        'axis': {'range': [0, 100]},
        'bar': {'color': '#2ec4b6'},
        'steps': [
            {'range': [0,  50], 'color': '#3d1c1c'},
            {'range': [50, 75], 'color': '#3d3219'},
            {'range': [75, 100],'color': '#1c3d2e'}
        ],
        'threshold': {'line': {'color': 'white', 'width': 3}, 'value': 75}
    },
    number={'suffix': '%'}
))
fig_gauge.update_layout(template='plotly_dark', width=450, height=360)
fig_gauge.show()

## 7. Scale Worms Over Time

In [ ]:
df_time = df[df['timestamp'].notna()].copy()

if df_time.empty:
    print('WARNING: No timestamps could be parsed from filenames.')
    print('Check your TIMESTAMP_REGEX setting in Cell 2.')
    print('Sample filename from dataset:', df['filename'].iloc[0] if not df.empty else 'none found')
else:
    df_time['date'] = df_time['timestamp'].dt.date

    daily = df_time.groupby('date').agg(
        total_detections =('filename',    'count'),
        confirmed_worms  =('is_scaleworm', 'sum'),
        false_positives  =('is_scaleworm', lambda x: (x == 0).sum())
    ).reset_index()
    daily['date'] = pd.to_datetime(daily['date'])

    print(f'Days with detection data: {len(daily)}')
    print(daily.to_string(index=False))

In [ ]:
if not df_time.empty:

    fig_time = make_subplots(
        rows=2, cols=1,
        subplot_titles=(
            'Confirmed Scale Worm Detections per Day',
            'All Detections: Confirmed Worms vs False Positives'
        ),
        vertical_spacing=0.16,
        shared_xaxes=True
    )

    # Top panel: confirmed worms over time
    fig_time.add_trace(
        go.Scatter(
            x=daily['date'],
            y=daily['confirmed_worms'],
            mode='lines+markers',
            name='Confirmed Scale Worms',
            line=dict(color='#2ec4b6', width=2.5),
            marker=dict(size=8),
            fill='tozeroy',
            fillcolor='rgba(46, 196, 182, 0.15)',
            hovertemplate='%{x|%b %d, %Y}<br>Confirmed worms: %{y}<extra></extra>'
        ), row=1, col=1
    )

    # Bottom panel: stacked bar (TP + FP per day)
    fig_time.add_trace(
        go.Bar(
            x=daily['date'], y=daily['confirmed_worms'],
            name='True Positives',
            marker_color='#2ec4b6',
            hovertemplate='%{x|%b %d}<br>True Positives: %{y}<extra></extra>'
        ), row=2, col=1
    )
    fig_time.add_trace(
        go.Bar(
            x=daily['date'], y=daily['false_positives'],
            name='False Positives',
            marker_color='#e76f51',
            hovertemplate='%{x|%b %d}<br>False Positives: %{y}<extra></extra>'
        ), row=2, col=1
    )

    fig_time.update_layout(
        title=dict(
            text=(
                '<i>Lepidonotopodium piscesae</i> — Abundance at Mushroom Vent, Axial Seamount<br>'
                '<sup>CAMHD OOI | October 2024</sup>'
            ),
            font=dict(size=15)
        ),
        template='plotly_dark',
        barmode='stack',
        height=660,
        legend=dict(orientation='h', y=-0.12),
        hovermode='x unified'
    )
    fig_time.update_yaxes(title_text='Worm Count',  row=1, col=1)
    fig_time.update_yaxes(title_text='Detections',  row=2, col=1)
    fig_time.update_xaxes(title_text='Date',        row=2, col=1)

    fig_time.show()

## 8. Summary Table

In [ ]:
date_range = (
    f"{df_time['timestamp'].min().strftime('%Y-%m-%d')} to "
    f"{df_time['timestamp'].max().strftime('%Y-%m-%d')}"
    if not df_time.empty else 'N/A'
)

summary = pd.DataFrame({
    'Metric': [
        'Total Detections Reviewed',
        'Confirmed Scale Worms (TP)',
        'False Positives (FP)',
        'Precision',
        'F1 Score',
        'Date Range'
    ],
    'Value': [
        total,
        tp,
        fp,
        f'{precision:.3f}',
        f'{f1:.3f}',
        date_range
    ]
})

fig_tbl = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Metric</b>', '<b>Value</b>'],
        fill_color='#264653',
        font=dict(color='white', size=13),
        align='left'
    ),
    cells=dict(
        values=[summary['Metric'], summary['Value']],
        fill_color=[['#1a1a2e' if i % 2 == 0 else '#16213e' for i in range(len(summary))]],
        font=dict(color='white', size=12),
        align='left',
        height=30
    )
)])
fig_tbl.update_layout(
    title='Summary — YOLO Mushroom Model | Lepidonotopodium piscesae',
    template='plotly_dark',
    width=620, height=340
)
fig_tbl.show()

print(summary.to_string(index=False))

---
## Notes

- **Precision** = TP / (TP + FP): the share of model detections that were genuinely scale worms. This is the key metric to track as you retrain.
- **Recall** cannot be computed from this dataset alone — you would need the total number of frames *without* a detection to know how many worms the model missed. If you have total frame counts, those can be added here.
- **Rerunning after each correction cycle:** replace the zip path in Cell 2 with your updated dataset and rerun all cells. Precision should trend upward over successive training iterations.
- **Extending to other species:** add additional folder name pairs to the loop in Cell 4, or expand the `POSITIVE_FOLDER` logic to handle multi-class datasets.